In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
from transformers import AlbertTokenizer
import math
from tqdm.notebook import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

tokenizer = AlbertTokenizer.from_pretrained('explosion-testing/albert-test')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
d_model = 128  # размерность скрытого пространства
num_heads = 8  # кол-во голов внимания
d_ff = 512  # размерность полносвязного слоя
num_layers = 4  # общее число слоев энкодера
lr = 1e-4  # скоростью обучения активации

In [35]:
def load_data(path):
    df = pd.read_csv(path, header=None, names=['l', 't', 's'], quotechar='"', engine='python', on_bad_lines='skip')
    df['l'] = df['l'].astype(int) - 1
    df['txt'] = df['t'].fillna('') + " " + df['s'].fillna('')
    return df

train_df = load_data('train.csv')
test_df = load_data('test.csv')

train_df = train_df.sample(20000, random_state=42).reset_index(drop=True)
test_df = test_df.sample(4000, random_state=42).reset_index(drop=True)

print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")

Train rows: 20000
Test rows: 4000


In [36]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

max_length = 128
batch_size = 32

train_dataset = ReviewDataset(train_df['txt'].values, train_df['l'].values, tokenizer, max_length)
test_dataset = ReviewDataset(test_df['txt'].values, test_df['l'].values, tokenizer, max_length)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"New size: {len(train_loader)} train batches, {len(test_loader)} test batches")

New size: 625 train batches, 125 test batches


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout = nn.Dropout(dropout)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous()
        out = out.view(batch_size, seq_len, self.d_model)
        out = self.out_proj(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        y = self.norm(x)
        y = sublayer(y)
        y = self.dropout(y)
        return x + y

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualBlock(d_model, dropout)
        self.residual2 = ResidualBlock(d_model, dropout)

    def forward(self, x, mask=None):
        x = self.residual1(x, lambda t: self.self_attn(t, mask))
        x = self.residual2(x, self.ffn)
        return x

class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(seq_len, device=x.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        token_embeddings = self.token_emb(x)
        position_embeddings = self.pos_emb(positions)


        return token_embeddings + position_embeddings

class EncoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len, dropout=0.1):
        super().__init__()
        self.embedding = TokenAndPositionEmbedding(vocab_size, d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.norm(x)
        x = x[:, 0, :]
        x = self.classifier(x)
        return x

vocab_size = tokenizer.vocab_size
num_classes = 2
max_len = 128
dropout = 0.1

model = EncoderOnlyTransformer(vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len, dropout).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

print(f"Model params: {sum(p.numel() for p in model.parameters())}")
print(f"Params: d_model={d_model}, num_heads={num_heads}, num_layers={num_layers}")

Model params: 4649986
Params: d_model=128, num_heads=8, num_layers=4


In [9]:
num_epochs = 3
train_losses = []
val_losses = []
val_accuracies = []
print(f"Epochs: {num_epochs}, Batch num: {len(train_loader)}, Device: {device}\n")
for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Test]'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(test_loader)
    val_losses.append(avg_val_loss)

    accuracy = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()
    val_accuracies.append(accuracy)

    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {accuracy:.4f}\n")

Epochs: 3, Batch num: 625, Device: cpu



Epoch 1/3 [Train]:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 1/3 [Test]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1: Train Loss: 0.6055, Val Loss: 0.4974, Val Acc: 0.7670



Epoch 2/3 [Train]:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 2/3 [Test]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 2: Train Loss: 0.4780, Val Loss: 0.4296, Val Acc: 0.8030



Epoch 3/3 [Train]:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 3/3 [Test]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 3: Train Loss: 0.4146, Val Loss: 0.3881, Val Acc: 0.8295



In [39]:
from transformers import AlbertForSequenceClassification, AlbertConfig
from torch.optim import AdamW

config = AlbertConfig.from_pretrained('explosion-testing/albert-test')
config.vocab_size = tokenizer.vocab_size
config.num_labels = 2

pretrained_model = AlbertForSequenceClassification.from_pretrained(
    'explosion-testing/albert-test',
    config=config,
    ignore_mismatched_sizes=True
).to(device)

optimizer_pt = AdamW(pretrained_model.parameters(), lr=lr)

print(f"ALBERT loaded. Params: {sum(p.numel() for p in pretrained_model.parameters())}")

Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: explosion-testing/albert-test
Key                                      | Status     |                                                                                              
-----------------------------------------+------------+----------------------------------------------------------------------------------------------
predictions.LayerNorm.bias               | UNEXPECTED |                                                                                              
predictions.bias                         | UNEXPECTED |                                                                                              
predictions.dense.bias                   | UNEXPECTED |                                                                                              
predictions.LayerNorm.weight             | UNEXPECTED |                                                                                              
predi

ALBERT loaded. Params: 3919879


In [40]:
num_epochs_pt = 3
train_losses_pt = []
val_losses_pt = []
val_accuracies_pt = []

print("fine-tuning ALBERT...")
print(f"Epoch: {num_epochs_pt}, Batch num: {len(train_loader)}, Device: {device}\n")

for epoch in range(num_epochs_pt):
    pretrained_model.train()
    total_train_loss = 0

    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs_pt} [Train ALBERT]')
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer_pt.zero_grad()
        outputs = pretrained_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer_pt.step()

        total_train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses_pt.append(avg_train_loss)

    pretrained_model.eval()
    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Epoch {epoch+1}/{num_epochs_pt} [Test ALBERT]'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = pretrained_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(test_loader)
    val_losses_pt.append(avg_val_loss)

    accuracy = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()
    val_accuracies_pt.append(accuracy)

    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {accuracy:.4f}\n")

print("Fine-tuning ALBERT finish.")

fine-tuning ALBERT...
Epoch: 3, Batch num: 625, Device: cpu



Epoch 1/3 [Train ALBERT]:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 1/3 [Test ALBERT]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1: Train Loss: 0.5436, Val Loss: 0.3308, Val Acc: 0.8708



Epoch 2/3 [Train ALBERT]:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 2/3 [Test ALBERT]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 2: Train Loss: 0.2559, Val Loss: 0.2982, Val Acc: 0.8805



Epoch 3/3 [Train ALBERT]:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 3/3 [Test ALBERT]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 3: Train Loss: 0.1644, Val Loss: 0.3387, Val Acc: 0.8695

Fine-tuning ALBERT finish.


In [38]:
def evaluate_model(model_obj, loader, is_hf=False):
    model_obj.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['label'].cpu().numpy()

            outputs = model_obj(input_ids=ids, attention_mask=mask)
            logits = outputs.logits if is_hf else outputs

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.extend(labels)
            y_pred.extend(preds)

    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred, average='weighted')
    }

metrics_custom = evaluate_model(model, test_loader, is_hf=False)
metrics_pt = evaluate_model(pretrained_model, test_loader, is_hf=True)

results_df = pd.DataFrame([metrics_custom, metrics_pt],
                          index=['Transformer', 'ALBERT (Fine-tuned)'])
results_df.index.name = 'Модель'

display(results_df.round(4))

,Accuracy,F1-score
Модель,,
Transformer,0.8295,0.8295
ALBERT (Fine-tuned),0.8672,0.8671
